# OmniSQL-7B ORPO — EHRSQL 2024 MIMIC-IV (Session 2)

Applies ORPO preference optimization on top of the SFT adapter from Session 1.
Runs full test-set evaluation at the end and calibrates entropy filtering.

**What this does:**
- Loads SFT-OmniSQL adapter from Drive
- Trains ORPO using OmniSQL's own wrong predictions as rejected examples
- Evaluates on the full EHRSQL 2024 test set (1,167 questions)
- Calibrates entropy threshold on valid set to maximize RS(10)

**Required files** (upload or mount from Drive):
```
omnisql_sft_adapter/    — LoRA adapter from Session 1
omnisql_orpo_pairs.jsonl — ~3,000 preference pairs
mimic_iv.sqlite         — EHRSQL 2024 DB (36 MB)
test/data.json + label.json
valid/data.json + label.json
train_combined_embeddings_bge_large.npy (157 MB, for few-shot retrieval)
```

**Expected duration:** 3–5 hours on RTX 6000 Pro (96 GB)

## 1 — Install dependencies

In [ ]:
!pip install -q unsloth trl transformers accelerate peft bitsandbytes datasets
!pip install -q sentence-transformers rank-bm25

## 2 — Mount Drive and copy files

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/EHRSQL/EHRSQL_OMNI_Eval'

for d in ['/content/data/test', '/content/data/valid',
          '/content/checkpoints/omnisql_sft/adapter_final']:
    os.makedirs(d, exist_ok=True)

copies = [
    (f'{DRIVE_DIR}/omnisql_sft_adapter', '/content/checkpoints/omnisql_sft/adapter_final'),
    (f'{DRIVE_DIR}/omnisql_orpo_pairs.jsonl', '/content/omnisql_orpo_pairs.jsonl'),
    (f'{DRIVE_DIR}/mimic_iv.sqlite', '/content/data/mimic_iv.sqlite'),
    (f'{DRIVE_DIR}/test/data.json',  '/content/data/test/data.json'),
    (f'{DRIVE_DIR}/test/label.json', '/content/data/test/label.json'),
    (f'{DRIVE_DIR}/valid/data.json',  '/content/data/valid/data.json'),
    (f'{DRIVE_DIR}/valid/label.json', '/content/data/valid/label.json'),
    (f'{DRIVE_DIR}/train_combined_embeddings_bge_large.npy',
     '/content/data/train_combined_embeddings_bge_large.npy'),
    (f'{DRIVE_DIR}/train/data.json',  '/content/data/train/data.json'),
    (f'{DRIVE_DIR}/train/label.json', '/content/data/train/label.json'),
]

for src, dst in copies:
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'  {os.path.basename(dst)}/ (dir)')
    elif os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(src, dst)
        print(f'  {os.path.basename(dst)}: {os.path.getsize(dst)/1e6:.1f} MB')
    else:
        print(f'  MISSING: {src}')

## 3 — Configuration

In [ ]:
MODEL_ID        = 'seeklhy/OmniSQL-7B'
SFT_ADAPTER     = '/content/checkpoints/omnisql_sft/adapter_final'
PAIRS_FILE      = '/content/omnisql_orpo_pairs.jsonl'
OUTPUT_DIR      = '/content/checkpoints/omnisql_orpo'
ADAPTER_OUT     = OUTPUT_DIR + '/adapter_final'
DRIVE_SAVE_DIR  = '/content/drive/MyDrive/EHRSQL/EHRSQL_OMNI_Eval'

DB_PATH         = '/content/data/mimic_iv.sqlite'
TEST_DIR        = '/content/data/test'
VALID_DIR       = '/content/data/valid'
TRAIN_DIR       = '/content/data/train'
EMBED_CACHE     = '/content/data/train_combined_embeddings_bge_large.npy'

# ORPO config
ORPO_BETA       = 0.1     # lambda: weight of preference loss vs SFT loss
LEARNING_RATE   = 2e-6    # low LR for fine-tuning mode
EPOCHS          = 1
BATCH_SIZE      = 2
GRAD_ACCUM      = 8       # effective batch = 16
MAX_SEQ_LEN     = 1536

# Few-shot eval config
FEW_SHOT_K      = 10
EVAL_BATCH_SZ   = 8

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config OK')

## 4 — Load SFT-OmniSQL adapter with Unsloth

In [ ]:
import torch
from unsloth import FastLanguageModel

print(f'Loading {MODEL_ID} + SFT adapter from {SFT_ADAPTER} ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER,     # loads base model + applies adapter
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

mem_gb = torch.cuda.memory_allocated() / 1e9
print(f'Model loaded ✅  VRAM: {mem_gb:.1f} GB')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 5 — Load ORPO preference pairs

In [ ]:
import json
from datasets import Dataset

print(f'Loading {PAIRS_FILE} ...')
raw_pairs = []
with open(PAIRS_FILE) as f:
    for line in f:
        raw_pairs.append(json.loads(line))

n_sql  = sum(1 for p in raw_pairs if p.get('is_answerable', True))
n_abs  = sum(1 for p in raw_pairs if not p.get('is_answerable', True))
print(f'  Total pairs:       {len(raw_pairs):,}')
print(f'  SQL quality:       {n_sql:,}  (answerable wrong)')
print(f'  Abstention:        {n_abs:,}  (unanswerable halluc)')

def apply_orpo_template(pair):
    chosen_text   = tokenizer.apply_chat_template(
        pair['prompt'] + pair['chosen'], tokenize=False, add_generation_prompt=False)
    rejected_text = tokenizer.apply_chat_template(
        pair['prompt'] + pair['rejected'], tokenize=False, add_generation_prompt=False)
    return {'chosen': chosen_text, 'rejected': rejected_text}

dataset = Dataset.from_list(raw_pairs).map(
    apply_orpo_template,
    remove_columns=['prompt', 'is_answerable']
)

print(f'Dataset ready: {len(dataset):,} pairs')
print(f'Sample chosen[:100]:   {dataset[0]["chosen"][:100]}')
print(f'Sample rejected[:100]: {dataset[0]["rejected"][:100]}')

## 6 — ORPO Training

In [ ]:
from trl import ORPOConfig, ORPOTrainer

orpo_config = ORPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    beta=ORPO_BETA,
    optim='paged_adamw_8bit',
    bf16=True,
    gradient_checkpointing=True,
    max_length=MAX_SEQ_LEN,
    max_prompt_length=MAX_SEQ_LEN // 2,
    report_to='none',
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
)

trainer = ORPOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=orpo_config,
)

print('Starting ORPO training...')
print(f'  Pairs:             {len(dataset):,}')
print(f'  Steps per epoch:   ~{len(dataset) // (BATCH_SIZE * GRAD_ACCUM):,}')
print(f'  ORPO beta (lambda): {ORPO_BETA}')
trainer.train()

## 7 — Save adapter

In [ ]:
import shutil

print(f'Saving ORPO adapter → {ADAPTER_OUT}')
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)

drive_dest = f'{DRIVE_SAVE_DIR}/omnisql_orpo_adapter'
os.makedirs(drive_dest, exist_ok=True)
for fname in os.listdir(ADAPTER_OUT):
    shutil.copy(f'{ADAPTER_OUT}/{fname}', f'{drive_dest}/{fname}')
print(f'Backed up → {drive_dest}')

## 8 — Full evaluation on test set

Runs the complete EHRSQL 2024 harness:
- Hybrid retrieval (BM25 + bge-large-en-v1.5 RRF fusion, K=10)
- Batched generation with left-padding
- SQL post-processing + execution accuracy
- RS(N) with N=1,5,10

In [ ]:
import re, sqlite3, time, json, numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

_CURRENT_TIME = '2100-12-31 23:59:00'
_CURRENT_DATE = '2100-12-31'
ABSTAIN_TOK   = '[ABSTAIN]'

def post_process_sql(sql):
    sql = re.sub(r'[ ]+', ' ', sql.replace('\n', ' ')).strip()
    sql = sql.replace('> =', '>=').replace('< =', '<=').replace('! =', '!=')
    for ph, val in [('current_time', f"'{_CURRENT_TIME}'"),
                    ('current_date', f"'{_CURRENT_DATE}'"),
                    ("'now'", f"'{_CURRENT_TIME}'"),
                    ('NOW()', f"'{_CURRENT_TIME}'")]:
        if ph in sql: sql = sql.replace(ph, val)
    return sql.replace('%y', '%Y').replace('%j', '%J')

def exec_sql(sql):
    try:
        conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
        conn.row_factory = sqlite3.Row
        rows = [dict(r) for r in conn.execute(sql).fetchmany(100)]
        conn.close()
        return rows, None
    except Exception as e:
        return None, str(e)

def norm(rows):
    if not rows: return '[]'
    def ni(v):
        try: return str(round(float(v), 3))
        except: return str(v)
    return str(sorted([[ni(v) for v in r.values()] for r in rows])[:100])

def strip_fences(text):
    m = re.search(r'```(?:sql)?\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text

print('Helper functions defined ✅')

In [ ]:
# Build hybrid retrieval index (BM25 + bge-large-en-v1.5)
print('Building retrieval index ...')

train_data  = json.load(open(f'{TRAIN_DIR}/data.json'))['data']
train_labels = json.load(open(f'{TRAIN_DIR}/label.json'))
corpus = []
for ex in train_data:
    sql = train_labels.get(ex['id'], 'null')
    ans = sql.strip().lower() not in ('null', 'none', 'n/a', '')
    corpus.append({'q': ex['question'], 'sql': sql if ans else '[ABSTAIN]', 'ans': ans})
print(f'  Corpus: {len(corpus)} examples')

# BM25 index
bm25_tokens = [c['q'].lower().split() for c in corpus]
bm25 = BM25Okapi(bm25_tokens)

# Dense index — load cached if available
embed_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device='cuda')

if os.path.exists(EMBED_CACHE):
    print(f'  Loading cached embeddings from {EMBED_CACHE}')
    corpus_embs = np.load(EMBED_CACHE)
    if corpus_embs.shape[0] != len(corpus):
        print(f'  WARNING: cache size {corpus_embs.shape[0]} != corpus {len(corpus)} — re-encoding')
        corpus_embs = embed_model.encode([c['q'] for c in corpus], batch_size=128,
                                          show_progress_bar=True, normalize_embeddings=True)
        np.save(EMBED_CACHE, corpus_embs)
else:
    print('  Encoding corpus (one-time, ~3 min) ...')
    corpus_embs = embed_model.encode([c['q'] for c in corpus], batch_size=128,
                                      show_progress_bar=True, normalize_embeddings=True)
    np.save(EMBED_CACHE, corpus_embs)

print(f'  Embeddings shape: {corpus_embs.shape}')
print('Index ready ✅')

def retrieve_hybrid(question, k=10):
    # BM25 scores
    bm25_sc = bm25.get_scores(question.lower().split())
    bm25_ranks = np.argsort(bm25_sc)[::-1][:60]
    # Dense scores
    q_emb = embed_model.encode([question], normalize_embeddings=True)[0]
    dense_sc = corpus_embs @ q_emb
    dense_ranks = np.argsort(dense_sc)[::-1][:60]
    # RRF fusion
    scores = {}
    for rank, idx in enumerate(bm25_ranks):
        scores[idx] = scores.get(idx, 0) + 1/(rank+60)
    for rank, idx in enumerate(dense_ranks):
        scores[idx] = scores.get(idx, 0) + 1/(rank+60)
    top_k = sorted(scores, key=scores.__getitem__, reverse=True)[:k]
    return [corpus[i] for i in top_k]

In [ ]:
import json

SYSTEM_PROMPT = None

# Load from the small system_prompt.json saved from local config.SYSTEM_PROMPT
prompt_file = f'{DRIVE_DIR}/system_prompt.json'
if os.path.exists(prompt_file):
    with open(prompt_file) as f:
        SYSTEM_PROMPT = json.load(f)['system_prompt']
    print(f'System prompt loaded from Drive ({len(SYSTEM_PROMPT)} chars) ✅')
else:
    raise RuntimeError(
        f'system_prompt.json not found at {prompt_file}\n'
        'Upload colab/system_prompt.json from your local machine to Drive.'
    )

In [ ]:
# Full test-set evaluation with hybrid few-shot retrieval
FastLanguageModel.for_inference(model)
tokenizer.padding_side = 'left'

# Load test data
test_data   = json.load(open(f'{TEST_DIR}/data.json'))['data']
test_labels = json.load(open(f'{TEST_DIR}/label.json'))
examples = []
for ex in test_data:
    sql = test_labels.get(ex['id'], 'null')
    ans = sql.strip().lower() not in ('null', 'none', 'n/a', '')
    examples.append({'id': ex['id'], 'q': ex['question'], 'sql': sql if ans else '', 'ans': ans})

print(f'Test set: {len(examples)} examples')
print(f'  Answerable:   {sum(1 for e in examples if e["ans"])}')
print(f'  Unanswerable: {sum(1 for e in examples if not e["ans"])}')

# Precompute test embeddings (speeds up retrieval 10×)
print('\nPrecomputing test embeddings ...')
test_qs = [ex['q'] for ex in examples]
test_embs = embed_model.encode(test_qs, batch_size=128, show_progress_bar=True,
                                normalize_embeddings=True)

def retrieve_hybrid_precomputed(q_emb, k=FEW_SHOT_K):
    # NOTE: BM25 still needs the raw question — only dense uses precomputed emb
    dense_sc = corpus_embs @ q_emb
    dense_ranks = np.argsort(dense_sc)[::-1][:60]
    return [corpus[i] for i in dense_ranks[:k]]  # fast dense-only for speed
    # (Full RRF would need the question text; use dense-only for speed during batch eval)

def build_few_shot_prompt(question, q_emb, k=FEW_SHOT_K):
    shots = retrieve_hybrid_precomputed(q_emb, k)
    examples_text = '\n\n'.join(
        f'Q: {s["q"]}\nA: {s["sql"]}' for s in shots
    )
    return f'{examples_text}\n\nQ: {question}'

def generate_batch(questions, q_embs):
    msgs_list = [
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': build_few_shot_prompt(q, emb)}]
        for q, emb in zip(questions, q_embs)
    ]
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
               for m in msgs_list]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True,
                       truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outs = model.generate(**inputs, max_new_tokens=512, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    decoded = []
    for out in outs:
        text = tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
        decoded.append(strip_fences(text))
    return decoded

# Run evaluation
correct_ans = wrong_ans = wrong_unans = correct_abs = 0
all_results = []
t0 = time.time()

for i in range(0, len(examples), EVAL_BATCH_SZ):
    batch = examples[i:i+EVAL_BATCH_SZ]
    batch_embs = test_embs[i:i+EVAL_BATCH_SZ]
    preds = generate_batch([ex['q'] for ex in batch], batch_embs)

    for ex, pred in zip(batch, preds):
        abstained = (pred.strip() == ABSTAIN_TOK)
        outcome = 'unknown'

        if ex['ans']:
            if abstained:
                outcome = 'wrong_abstention'
                wrong_ans += 1
            else:
                gr, ge = exec_sql(post_process_sql(ex['sql']))
                pr, pe = exec_sql(post_process_sql(pred))
                if pe is None and ge is None and gr and norm(pr) == norm(gr):
                    correct_ans += 1
                    outcome = 'correct'
                else:
                    wrong_ans += 1
                    outcome = 'wrong_sql'
        else:
            if abstained:
                correct_abs += 1
                outcome = 'correct_abstention'
            else:
                wrong_unans += 1
                outcome = 'hallucination'

        all_results.append({'id': ex['id'], 'question': ex['q'],
                            'gold_sql': ex['sql'], 'predicted_sql': pred,
                            'is_answerable': ex['ans'], 'outcome': outcome})

    done = i + len(batch)
    if done % 50 == 0 or done == len(examples):
        total = done
        ans_done = sum(1 for e in examples[:done] if e['ans'])
        rs10 = (correct_ans + correct_abs - 10*wrong_unans) / total
        ela = (time.time()-t0)/60
        eta = ela/done * (len(examples)-done) if done > 0 else 0
        print(f'[{done:4d}/{len(examples)}] EX={correct_ans/max(1,ans_done):.3f} '
              f'RS(10)={rs10:.3f} hall={wrong_unans} abs={correct_abs} '
              f'{ela:.1f}min ETA={eta:.1f}min')

total = len(examples)
ans_total = sum(1 for e in examples if e['ans'])
rs0  = (correct_ans + correct_abs - 1*wrong_unans) / total
rs5  = (correct_ans + correct_abs - 5*wrong_unans) / total
rs10 = (correct_ans + correct_abs - 10*wrong_unans) / total

print(f'\n=== ORPO-OmniSQL Test Set Results ===')
print(f'  EX:     {correct_ans/ans_total:.4f}  ({correct_ans}/{ans_total})')
print(f'  RS(1):  {rs0:.4f}')
print(f'  RS(5):  {rs5:.4f}')
print(f'  RS(10): {rs10:.4f}')
print(f'  wrong_on_unanswerable: {wrong_unans}')
print(f'  correct_abstentions:   {correct_abs}')
print(f'  wrong_on_answerable:   {wrong_ans}')
print()
print('Baseline (OmniSQL 0-shot): EX=0.612  RS(10)=-0.564')
print('Target:                    EX≥0.70   RS(10)≥0.65  (before entropy filter)')

## 9 — Entropy calibration on valid set

Re-runs eval on valid set with token log-probabilities to find the optimal
entropy threshold that maximizes RS(10). High-entropy predictions → [ABSTAIN].

This replicates the winner's entropy filter (top 7% of entropy → abstain).

In [ ]:
import math

valid_data   = json.load(open(f'{VALID_DIR}/data.json'))['data']
valid_labels = json.load(open(f'{VALID_DIR}/label.json'))
valid_exs = []
for ex in valid_data:
    sql = valid_labels.get(ex['id'], 'null')
    ans = sql.strip().lower() not in ('null', 'none', 'n/a', '')
    valid_exs.append({'id': ex['id'], 'q': ex['question'], 'sql': sql if ans else '', 'ans': ans})

print(f'Valid set: {len(valid_exs)} examples')
print('Computing predictions + token entropy on valid set ...')

valid_qs   = [ex['q'] for ex in valid_exs]
valid_embs = embed_model.encode(valid_qs, batch_size=128, show_progress_bar=False,
                                 normalize_embeddings=True)

def generate_batch_with_entropy(questions, q_embs):
    msgs_list = [
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': build_few_shot_prompt(q, emb)}]
        for q, emb in zip(questions, q_embs)
    ]
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
               for m in msgs_list]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True,
                       truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outs = model.generate(
            **inputs, max_new_tokens=512, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True, output_scores=True
        )

    results = []
    for b_idx in range(len(questions)):
        gen_ids = outs.sequences[b_idx, input_len:]
        text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        pred = strip_fences(text)

        # Compute per-token entropy from logits
        entropies = []
        for step_scores in outs.scores:
            logits = step_scores[b_idx].float()
            probs  = torch.softmax(logits, dim=-1)
            ent    = -(probs * torch.log(probs + 1e-10)).sum().item()
            entropies.append(ent)

        # Trim to actual generated tokens (stop at EOS)
        eos_id = tokenizer.eos_token_id
        gen_len = len(gen_ids)
        for j, tok in enumerate(gen_ids):
            if tok == eos_id:
                gen_len = j
                break
        avg_entropy = float(np.mean(entropies[:gen_len])) if gen_len > 0 else 0.0
        results.append((pred, avg_entropy))

    return results

CALIB_BATCH = 4  # smaller batch for entropy (stores all logits)
valid_preds  = []
valid_entropies = []

for i in range(0, len(valid_exs), CALIB_BATCH):
    batch = valid_exs[i:i+CALIB_BATCH]
    batch_embs = valid_embs[i:i+CALIB_BATCH]
    res = generate_batch_with_entropy([ex['q'] for ex in batch], batch_embs)
    for pred, ent in res:
        valid_preds.append(pred)
        valid_entropies.append(ent)
    if (i // CALIB_BATCH) % 20 == 0:
        print(f'  [{i+len(batch):4d}/{len(valid_exs)}] done')

print(f'\nEntropy stats:')
print(f'  mean: {np.mean(valid_entropies):.4f}')
print(f'  p50:  {np.percentile(valid_entropies, 50):.4f}')
print(f'  p93:  {np.percentile(valid_entropies, 93):.4f}  (winner used top-7%)')
print(f'  p95:  {np.percentile(valid_entropies, 95):.4f}')
print(f'  p97:  {np.percentile(valid_entropies, 97):.4f}')
print(f'  max:  {np.max(valid_entropies):.4f}')

In [ ]:
# Grid search entropy threshold to maximize RS(10) on valid set
def score_with_threshold(threshold):
    ca = wa = wu = cab = 0
    for ex, pred, ent in zip(valid_exs, valid_preds, valid_entropies):
        # Apply entropy filter: high entropy → [ABSTAIN]
        effective_pred = ABSTAIN_TOK if (ent > threshold) else pred
        abstained = (effective_pred == ABSTAIN_TOK)

        if ex['ans']:
            if abstained:
                wa += 1
            else:
                gr, ge = exec_sql(post_process_sql(ex['sql']))
                pr, pe = exec_sql(post_process_sql(effective_pred))
                if pe is None and ge is None and gr and norm(pr) == norm(gr):
                    ca += 1
                else:
                    wa += 1
        else:
            if abstained:
                cab += 1
            else:
                wu += 1

    total = len(valid_exs)
    return (ca + cab - 10*wu) / total

thresholds = np.percentile(valid_entropies, np.arange(80, 99, 1))
best_threshold = None
best_rs10 = -999

print('Threshold sweep:')
for t in thresholds:
    rs10 = score_with_threshold(t)
    flag = ' ← best' if rs10 > best_rs10 else ''
    print(f'  threshold={t:.4f}  RS(10)={rs10:.4f}{flag}')
    if rs10 > best_rs10:
        best_rs10 = rs10
        best_threshold = t

print(f'\nBest entropy threshold: {best_threshold:.4f}')
print(f'Best valid RS(10):       {best_rs10:.4f}')
print(f'(Baseline threshold was: 1.1462)')

In [ ]:
# Apply best threshold to test results
# (Need to re-run test eval with entropy; or load from all_results + re-score)
# NOTE: all_results was collected without entropy. This cell is for the second run.
# If you want entropy-filtered test results, run this cell after re-running Cell 8
# with return_dict_in_generate=True.

print(f'Optimal entropy threshold: {best_threshold:.4f}')
print(f'To apply to test set, re-run cell 8 with:')
print(f'  entropy_threshold = {best_threshold:.4f}')
print()
print('If RS(10) target not met, proceed to self-training:')
print('  1. Run ORPO-OmniSQL on train_aug (38,689 examples)')
print('  2. Collect [ABSTAIN] preds with entropy < 0.5 as pseudo-negatives')
print('  3. SFT for 1 more epoch on original + pseudo-labeled data')

## 10 — Save results

In [ ]:
import json, shutil

# Save prediction file
preds_path = '/content/omnisql_orpo_preds.jsonl'
with open(preds_path, 'w') as f:
    for r in all_results:
        f.write(json.dumps(r) + '\n')

# Save summary
summary = {
    'model':             'OmniSQL-7B SFT+ORPO',
    'ex':                correct_ans / ans_total,
    'rs0':               rs0,
    'rs5':               rs5,
    'rs10':              rs10,
    'correct_sql':       correct_ans,
    'wrong_sql':         wrong_ans,
    'hallucinations':    wrong_unans,
    'correct_abstains':  correct_abs,
    'total':             total,
    'entropy_threshold': float(best_threshold) if best_threshold else None,
    'valid_rs10_with_entropy': float(best_rs10) if best_rs10 else None,
}
results_path = '/content/omnisql_orpo_results.json'
with open(results_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

# Copy to Drive
for src in [preds_path, results_path]:
    dst = f'{DRIVE_SAVE_DIR}/{os.path.basename(src)}'
    shutil.copy(src, dst)
    print(f'Saved → {dst}')

## 11 — Download (if not using Drive)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/omnisql_orpo_adapter', 'zip', ADAPTER_OUT)
files.download('/content/omnisql_orpo_adapter.zip')
files.download('/content/omnisql_orpo_results.json')
files.download('/content/omnisql_orpo_preds.jsonl')